# 0.0 Graph Sanity Check

**Learning:**
- [0.0 Getting Started](../../../Learning/LangGraph/00_foundations/0.0_getting_started.md)
- [1.1 Mental Model](../../../Learning/LangGraph/00_foundations/1.1_mental_model.md)

**Goal:** Confirm LangGraph is installed, build your first `StateGraph`, and compare it side-by-side with an LCEL pipeline.

## Setup

In [ ]:
import sys
from pathlib import Path

# Project root (repo root when running in Docker/Jupyter)
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

## 1. Verify LangGraph Installation

In [ ]:
import langgraph
from langgraph.graph import StateGraph, START, END

print(f"LangGraph version: {langgraph.__version__}")
print("Imports OK: StateGraph, START, END")

## 2. Your First StateGraph

A graph has three parts:
1. **State** — typed data shared between nodes
2. **Nodes** — functions that read and update state
3. **Edges** — connections that define execution order

Flow: `START → greet → END`

In [ ]:
from typing import TypedDict


class State(TypedDict):
    message: str


def greet(state: State) -> State:
    return {"message": f"Hello, {state['message']}!"}


graph = StateGraph(State)
graph.add_node("greet", greet)
graph.add_edge(START, "greet")
graph.add_edge("greet", END)

app = graph.compile()
result = app.invoke({"message": "LangGraph"})
result

## 3. LCEL vs. StateGraph (Same Transform, Different Model)

| LCEL | LangGraph |
|---|---|
| Linear pipeline | Explicit nodes and edges |
| Great for stateless transforms | Great when steps branch, loop, or persist |

Both approaches below apply the same greeting transform.

In [ ]:
from langchain_core.runnables import RunnableLambda

# LCEL: pipe-style transformation
lcel_chain = RunnableLambda(lambda x: {"message": f"Hello, {x['message']}!"})
lcel_result = lcel_chain.invoke({"message": "LangGraph"})

print("LCEL result:", lcel_result)
print("Graph result:", result)
print("Same output:", lcel_result == result)

## 4. Optional — LCEL with LLM vs. Graph with LLM Node

Skip this cell if `OPENAI_API_KEY` is not configured. It shows how an LLM fits **inside** a graph node—the pattern used throughout this track.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from src.config import OPENAI_API_KEY
from src.llms.openai_chat import make_openai_chat

if not OPENAI_API_KEY:
    print("Skipping LLM demo — set OPENAI_API_KEY in .env to run this cell.")
else:
    llm = make_openai_chat()
    prompt = ChatPromptTemplate.from_template(
        "Say hello to {name} in one short sentence."
    )

    # LCEL pipeline
    lcel_llm_chain = prompt | llm | StrOutputParser()
    lcel_reply = lcel_llm_chain.invoke({"name": "LangGraph"})
    print("LCEL + LLM:", lcel_reply)

    # LangGraph: LLM as a node
    class LlmState(TypedDict):
        name: str
        reply: str

    def llm_node(state: LlmState) -> LlmState:
        reply = lcel_llm_chain.invoke({"name": state["name"]})
        return {"reply": reply}

    llm_graph = StateGraph(LlmState)
    llm_graph.add_node("llm", llm_node)
    llm_graph.add_edge(START, "llm")
    llm_graph.add_edge("llm", END)

    llm_app = llm_graph.compile()
    graph_reply = llm_app.invoke({"name": "LangGraph", "reply": ""})
    print("Graph + LLM:", graph_reply["reply"])

## Exit Criteria Checklist

- [ ] LangGraph imports successfully
- [ ] You ran `START → greet → END` and got a transformed message
- [ ] You can explain why a graph is better than a `while` loop for agent control
- [ ] You see that LCEL and LangGraph solve different problems—and work together

**Next:** [1.1 First StateGraph](../../../Learning/LangGraph/01_beginner/1.1_first_stategraph.md) → `../01_beginner/1.1_first_stategraph.ipynb`